# ⚖️ Thực Nghiệm 04: Kỹ Thuật Xử Lý Dữ Liệu Mất Cân Bằng (Imbalanced Data)
**Quy tắc nghiệp vụ**: `BR-06: SMOTE Train-Only Validation` — Chống rò rỉ dữ liệu.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.metrics import precision_recall_curve, auc
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from scripts.train_xgboost_real import generate_training_data
X, y = generate_training_data(n_normal=25000, n_fraud=500)
print(f'Mẫu: {len(y)} | Gian lận: {(y == 1).sum()}')


### 2. K-Fold CV Chứng minh BR-06


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
res = {'Baseline': [], 'scale_pos_weight': [], 'SMOTE (Train-Only)': [], 'Under-Sampling': []}
for tr_idx, val_idx in skf.split(X, y):
    X_tr, y_tr, X_v, y_v = X[tr_idx], y[tr_idx], X[val_idx], y[val_idx]
    # 1. Base
    m1 = XGBClassifier(n_estimators=80, max_depth=5, eval_metric='aucpr', random_state=42).fit(X_tr, y_tr)
    p1, r1, _ = precision_recall_curve(y_v, m1.predict_proba(X_v)[:, 1]); res['Baseline'].append(auc(r1, p1))
    # 2. Scale pos weight
    spw = (len(y_tr)-y_tr.sum())/y_tr.sum()
    m2 = XGBClassifier(n_estimators=80, max_depth=5, scale_pos_weight=spw, eval_metric='aucpr', random_state=42).fit(X_tr, y_tr)
    p2, r2, _ = precision_recall_curve(y_v, m2.predict_proba(X_v)[:, 1]); res['scale_pos_weight'].append(auc(r2, p2))
    # 3. SMOTE Train-Only
    X_sm, y_sm = SMOTE(random_state=42).fit_resample(X_tr, y_tr)
    m3 = XGBClassifier(n_estimators=80, max_depth=5, eval_metric='aucpr', random_state=42).fit(X_sm, y_sm)
    p3, r3, _ = precision_recall_curve(y_v, m3.predict_proba(X_v)[:, 1]); res['SMOTE (Train-Only)'].append(auc(r3, p3))
    # 4. Under-sampling
    X_u, y_u = RandomUnderSampler(random_state=42).fit_resample(X_tr, y_tr)
    m4 = XGBClassifier(n_estimators=80, max_depth=5, eval_metric='aucpr', random_state=42).fit(X_u, y_u)
    p4, r4, _ = precision_recall_curve(y_v, m4.predict_proba(X_v)[:, 1]); res['Under-Sampling'].append(auc(r4, p4))
pd.DataFrame([{'Method': k, 'Mean PR-AUC': np.mean(v), 'Std': np.std(v)} for k, v in res.items()])
